# EV Charging Dataset — Data Cleaning & Quality Checks

This notebook prepares the EV charging records for EDA and machine-learning work.

The main checks performed in this notebook are:
- dataset size and structure
- missing values
- categorical consistency
- date/time reconstruction
- duration validation
- exact duplicates
- invalid zero/negative durations
- possible duplicate charging sessions
- duration and demand outliers
- basic identifier and chronology validation
- final export of the cleaned dataset



## Step 1 — Load the dataset and understand its structure

### What this step does
The CSV file is loaded into a Pandas DataFrame and inspect:
- the number of rows and columns
- column names
- data types
- non-null counts
- first few records
- missing values

### Column-name standardization
The column names are converted to lowercase and leading/trailing spaces are removed. This makes column references consistent and reduces typing errors later.

### Location standardization
`location` is converted to lowercase and extra spaces are removed.

**Example:**  
`" Hotel "`, `"HOTEL"` and `"hotel"` should represent the same category.


In [5]:
import pandas as pd

# Load dataset
df = pd.read_csv('data/ChargingRecords.csv')

# 1. Basic overview
print("--- INITIAL SHAPE ---")
print(df.shape)

# 2. Standardize column names to snake_case
df.columns = df.columns.str.lower().str.strip()
# Let's verify the renamed columns
print("Updated Column Names:")
print(df.columns.tolist())

# 3. Check info and records structure and missing values
print("\n--- INFO ---")
print(df.info())

print("\n--- HEAD ---")
display(df.head())

print("\n--- MISSING VALUES ---")
print(df.isnull().sum())

# 4. Convert numeric columns to numbers
# If an invalid value is found, errors='coerce' changes it to NaN.
numeric_columns = [
    "userid",
    "chargerid",
    "chargercompany",
    "chargertype",
    "duration",
    "demand",
]

for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

# 5. Clean the location column
# strip() removes spaces before and after the text.
# lower() changes HOTEL and Hotel to hotel.
df["location"] = df["location"].astype("string").str.strip().str.lower()

--- INITIAL SHAPE ---
(72856, 13)
Updated Column Names:
['userid', 'chargerid', 'chargercompany', 'location', 'chargertype', 'startday', 'starttime', 'endday', 'endtime', 'startdatetime', 'enddatetime', 'duration', 'demand']

--- INFO ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 72856 entries, 0 to 72855
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   userid          72856 non-null  int64  
 1   chargerid       72856 non-null  int64  
 2   chargercompany  72856 non-null  int64  
 3   location        72856 non-null  object 
 4   chargertype     72856 non-null  int64  
 5   startday        72856 non-null  object 
 6   starttime       72856 non-null  object 
 7   endday          72856 non-null  object 
 8   endtime         72856 non-null  object 
 9   startdatetime   72856 non-null  object 
 10  enddatetime     72856 non-null  object 
 11  duration        72856 non-null  int64  
 12  demand          7

,userid,chargerid,chargercompany,location,chargertype,startday,starttime,endday,endtime,startdatetime,enddatetime,duration,demand
0,0,1,1,hotel,0,2022-09-15,20:54:02,2022-09-15,23:59:13,2022-09-15 20:54,2022-09-15 23:59,185,20.36
1,0,1,1,hotel,0,2022-09-14,20:01:05,2022-09-14,21:31:04,2022-09-14 20:01,2022-09-14 21:31,90,10.19
2,0,1,1,hotel,0,2022-09-14,18:54:30,2022-09-14,19:54:29,2022-09-14 18:54,2022-09-14 19:54,60,6.78
3,0,1,1,hotel,0,2022-09-29,18:32:51,2022-09-30,0:16:42,2022-09-29 18:32,2022-09-30 0:16,344,37.65
4,0,1,1,hotel,0,2022-09-25,19:30:15,2022-09-26,0:30:14,2022-09-25 19:30,2022-09-26 0:30,300,33.81



--- MISSING VALUES ---
userid            0
chargerid         0
chargercompany    0
location          0
chargertype       0
startday          0
starttime         0
endday            0
endtime           0
startdatetime     0
enddatetime       0
duration          0
demand            0
dtype: int64


## Step 2 — Check categorical and identifier values

### What this step does
We inspect the unique values of:
- `location`
- `chargertype`
- `chargercompany`

We also count the number of unique:
- `chargerid`
- `userid`

### Why this is required
Unique-value checks help identify unexpected category values, spelling differences, or invalid codes.

`UserID` and `ChargerID` are identifiers and are expected to repeat because:
- one user can have multiple charging sessions
- one charger can serve multiple users/sessions

In [6]:
# 5. Unique values check
print("--- UNIQUE VALUES ---")
print("Location unique count & values:", df['location'].unique().tolist())
print("\nChargerType unique values:", df['chargertype'].unique().tolist()) 
print("\nChargerCompany unique values:", df['chargercompany'].unique())
print(f"\nTotal Unique ChargerIDs: {df['chargerid'].nunique()}")
print(f"\nTotal Unique UserIDs: {df['userid'].nunique()}")

--- UNIQUE VALUES ---
Location unique count & values: ['hotel', 'company', 'apartment', 'resort', 'public area', 'public institution', 'restaurant', 'accommodation', 'golf', 'sightseeing', 'market', 'camping', 'public parking lot', 'bus garage']

ChargerType unique values: [0, 1]

ChargerCompany unique values: [1 0]

Total Unique ChargerIDs: 2119

Total Unique UserIDs: 2337


## Step 3 — Reconstruct accurate charging timestamps

### What this step does
The notebook combines:
- `startday + starttime` → `start_full`
- `endday + endtime` → `end_full`

These reconstructed timestamps preserve the full date and time information available in the original separate columns.

`errors="coerce"` converts an unparseable timestamp into `NaT`, which allows us to identify invalid date/time records instead of stopping execution with an error.

### Why this is required
A charging session may cross midnight.

**Example:**  
Start: `2022-06-10 23:50:00`  
End: `2022-06-11 01:20:00`

Comparing only the clock time would incorrectly suggest that the end time is earlier. Full timestamps allow us to handle such sessions correctly.

## Step 4 — Validate and recalculate charging duration

### What this step does
The original `duration` is first saved as `duration_original`.

A new duration is then calculated from:

`end_full - start_full`

and converted into minutes.

The notebook creates `duration_mismatch` to show whether the stored duration differs from the duration calculated from timestamps.

Finally, the `duration` column is replaced with the recalculated value.

### Why this is required
Duration should be logically consistent with the session's start and end timestamps. Recalculating it provides an independent validation of the original duration values.

### Why keep the original duration?
Keeping `duration_original` allows us to compare:
- source value
- calculated value

This is useful for auditability and explaining what was corrected.


## Step 5 — Remove exact duplicate records

### What this step does
`df.duplicated()` identifies rows where **all column values are identical**.

The duplicate count is printed and `drop_duplicates(keep="first")` keeps one copy while removing repeated copies.

### Why this is required
Exact duplicate records can cause the same charging session to be counted multiple times. This can distort:
- session counts
- averages
- total demand
- EDA results
- model training

In [7]:
# 6. Create accurate start and end timestamps by using (startday + starttime) and (endday + endtime)
df["start_full"] = pd.to_datetime(
    df["startday"].astype("string").str.strip()
    + " "
    + df["starttime"].astype("string").str.strip(),
    errors="coerce"
)
df["end_full"] = pd.to_datetime(
    df["endday"].astype("string").str.strip()
    + " "
    + df["endtime"].astype("string").str.strip(),
    errors="coerce"
)
# Check invalid timestamps
print("Invalid start timestamps:", df["start_full"].isnull().sum())
print("Invalid end timestamps:", df["end_full"].isnull().sum())

# 7. Recalculate duration from the timestamps
# First save the old duration so we can compare it later
df["duration_original"] = df["duration"]

# Calculate duration in minutes
df["duration_calculated"] = ((df["end_full"] - df["start_full"]).dt.total_seconds() / 60).round()

# True means old duration and calculated duration are different
df["duration_mismatch"] = (df["duration_original"] != df["duration_calculated"])
print("Duration mismatches:", df["duration_mismatch"].sum())
# Replace the old duration with recalculated duration
df["duration"] = df["duration_calculated"]

# 8. Remove exact duplicate rows
exact_duplicate_count = df.duplicated().sum()
print("Exact duplicate rows:", exact_duplicate_count)
df = df.drop_duplicates(keep="first")

Invalid start timestamps: 0
Invalid end timestamps: 0
Duration mismatches: 5676
Exact duplicate rows: 13


## Step 6 — Check possible duplicate sessions using a composite combination

### What this step does
checks repeated combinations of:
- `userid`
- `chargerid`
- `startdatetime`

In [8]:
# 9. Remove composite primary key
# Primary key not present in dataset, so same user cannot do charging on same time at same location
# Keep duplicates=False so that all instances of the duplicated primary key rows are shown
duplicate_pk_rows = df[df.duplicated(subset=['userid', 'chargerid', 'startdatetime'], keep=False)]
print(f"Total duplicate rows based on PK: {len(duplicate_pk_rows)}")

# Display the rows sorted by userid and startdatetime to easily compare them side-by-side
display(duplicate_pk_rows.sort_values(by=['userid', 'startdatetime']))

Total duplicate rows based on PK: 42


,userid,chargerid,chargercompany,location,chargertype,startday,starttime,endday,endtime,startdatetime,enddatetime,duration,demand,start_full,end_full,duration_original,duration_calculated,duration_mismatch
19535,0,118,1,public institution,0,2021-10-21,21:45:40,2021-10-22,0:15:45,2021-10-21 21:45,2021-10-22 0:15,150.0,13.90,2021-10-21 21:45:40,2021-10-22 00:15:45,140,150.0,True
19536,0,118,1,public institution,0,2021-10-21,21:45:40,2021-10-22,0:05:44,2021-10-21 21:45,2021-10-22 0:05,140.0,13.90,2021-10-21 21:45:40,2021-10-22 00:05:44,0,140.0,True
11967,0,59,1,apartment,0,2021-11-19,22:04:23,2021-11-19,23:00:30,2021-11-19 22:04,2021-11-19 23:00,56.0,6.60,2021-11-19 22:04:23,2021-11-19 23:00:30,56,56.0,False
11968,0,59,1,apartment,0,2021-11-19,22:04:50,2021-11-20,2:02:49,2021-11-19 22:04,2021-11-20 2:02,238.0,28.20,2021-11-19 22:04:50,2021-11-20 02:02:49,238,238.0,False
7683,0,30,1,resort,0,2022-02-03,9:44:09,2022-02-03,9:44:14,2022-02-03 9:44,2022-02-03 9:44,0.0,0.01,2022-02-03 09:44:09,2022-02-03 09:44:14,0,0.0,False
7684,0,30,1,resort,0,2022-02-03,9:44:49,2022-02-03,10:44:31,2022-02-03 9:44,2022-02-03 10:44,60.0,6.80,2022-02-03 09:44:49,2022-02-03 10:44:31,60,60.0,False
26972,0,194,1,company,0,2022-02-21,13:42:37,2022-02-21,14:28:20,2022-02-21 13:42,2022-02-21 14:28,46.0,5.30,2022-02-21 13:42:37,2022-02-21 14:28:20,0,46.0,True
26973,0,194,1,company,0,2022-02-21,13:42:37,2022-02-21,13:42:38,2022-02-21 13:42,2022-02-21 13:42,0.0,5.30,2022-02-21 13:42:37,2022-02-21 13:42:38,46,0.0,True
23804,0,163,1,resort,0,2022-04-19,21:31:37,2022-04-19,21:34:11,2022-04-19 21:31,2022-04-19 21:34,3.0,0.05,2022-04-19 21:31:37,2022-04-19 21:34:11,3,3.0,False
23805,0,163,1,resort,0,2022-04-19,21:31:55,2022-04-20,6:31:54,2022-04-19 21:31,2022-04-20 6:31,540.0,13.67,2022-04-19 21:31:55,2022-04-20 06:31:54,540,540.0,False


## Step 7 — Remove repeated composite-session records

### What this step does
For repeated `userid + chargerid + startdatetime` combinations, the notebook keeps the first record and removes later occurrences.

### Reason used in this notebook
The intention is to prevent the same apparent charging event from being represented more than once.

### Important assumption to mention
This removal depends on that one user using the same charger at the same recorded start time represents one unique charging session.

In [9]:
# Drop duplicates based on the primary key combination, keeping the first occurrence
df = df.drop_duplicates(subset=['userid', 'chargerid', 'startdatetime'], keep='first')
print(f"Total rows after dropping primary key duplicates: {len(df)}")

Total rows after dropping primary key duplicates: 72822


## Step 8 — Identify overnight charging sessions

### What this step does
Rows where `startday` and `endday` are different are counted as overnight/multi-day sessions.

### Why this is required
Different start and end dates are **not automatically an error**.

A user can start charging late at night and finish after midnight.

**Example:**  
Start: 11:30 PM  
End: 1:00 AM the next day

This check helps us distinguish valid overnight sessions from genuinely invalid chronology.

## Step 9 — Check zero, negative, and non-positive values

### What this step does
The notebook checks:
- `duration == 0`
- `duration < 0`
- `demand <= 0`

The current cleaning rule then keeps only rows where `duration > 0`.

### Why this check is important
- A **negative duration** is logically invalid because a charging session cannot have negative elapsed time.
- A **zero duration** may represent an immediate disconnect, logging issue, or very short event depending on the dataset/business definition.
- Non-positive demand should also be inspected because it may represent an invalid or non-charging event.

Treats zero and negative duration rows as invalid/system-glitch records and removes them.

In [10]:
# 9. Overnight sessions check (Start date != End date)
overnight_sessions = df[df['startday'] != df['endday']]
print(f"Overnight Sessions Count: {len(overnight_sessions)}")

# 10. Check invalid records and remove it.
print("Found Zero Duration rows:", (df['duration'] == 0).sum())
print("Found Negative Duration rows:", (df['duration'] < 0).sum())
print("Zero/Negative Demand Rows:", (df['demand'] <= 0).sum())

# 11. check invalid charger_company,chargertype, negative_user_id and negative_charger_id
invalid_charger_company = ~df["chargercompany"].isin([0, 1])
invalid_charger_type = ~df["chargertype"].isin([0, 1])
negative_user_id = df["userid"] < 0
negative_charger_id = df["chargerid"] < 0
missing_duration = df["duration"].isnull()

# 12. Required values must be present after numeric conversion.
required_value_columns = [
    "userid",
    "chargerid",
    "chargercompany",
    "location",
    "chargertype",
    "duration",
    "demand",
]
missing_required_value = df[required_value_columns].isnull().any(axis=1)

print("Invalid charger company values:", invalid_charger_company.sum())
print("Invalid charger type values:", invalid_charger_type.sum())
print("Negative user IDs:", negative_user_id.sum())
print("Negative charger IDs:", negative_charger_id.sum())
print("Missing durations:", missing_duration.sum())
print("Missing required values:", missing_required_value.sum())

# 13. Drop the negative/zero duration - Since it is a system glitch
df = df[df['duration'] > 0]
print(f"Final rows after selective update: {len(df)}")

Overnight Sessions Count: 10817
Found Zero Duration rows: 564
Found Negative Duration rows: 17
Zero/Negative Demand Rows: 0
Invalid charger company values: 0
Invalid charger type values: 0
Negative user IDs: 0
Negative charger IDs: 0
Missing durations: 0
Missing required values: 0
Final rows after selective update: 72241


## Step 10 — Identify potential outliers using the IQR method

This notebook checks outliers for:
- `duration`
- `demand`

In [11]:
# 13. Find duration outliers using the IQR method
# Outliers are flagged, not deleted.

duration_q1 = df["duration"].quantile(0.25)
duration_q3 = df["duration"].quantile(0.75)
duration_iqr = duration_q3 - duration_q1

duration_lower_limit = duration_q1 - 1.5 * duration_iqr
duration_upper_limit = duration_q3 + 1.5 * duration_iqr

df["duration_outlier"] = (
    (df["duration"] < duration_lower_limit)
    | (df["duration"] > duration_upper_limit)
)

print("\nDuration outlier information")
print("Q1:", duration_q1)
print("Q3:", duration_q3)
print("IQR:", duration_iqr)
print("Lower limit:", duration_lower_limit)
print("Upper limit:", duration_upper_limit)
print("Duration outliers:", df["duration_outlier"].sum())

# 14. Find demand outliers using the IQR method
# Again, these rows are flagged but not deleted.

demand_q1 = df["demand"].quantile(0.25)
demand_q3 = df["demand"].quantile(0.75)
demand_iqr = demand_q3 - demand_q1

demand_lower_limit = demand_q1 - 1.5 * demand_iqr
demand_upper_limit = demand_q3 + 1.5 * demand_iqr

df["demand_outlier"] = (
    (df["demand"] < demand_lower_limit)
    | (df["demand"] > demand_upper_limit)
)

print("\nDemand outlier information")
print("Q1:", demand_q1)
print("Q3:", demand_q3)
print("IQR:", demand_iqr)
print("Lower limit:", demand_lower_limit)
print("Upper limit:", demand_upper_limit)
print("Demand outliers:", df["demand_outlier"].sum())


# 15. Look at some outlier records
print("\nSome duration outliers:")
print(
    df[df["duration_outlier"]][
        ["userid", "location", "duration", "demand"]
    ].head()
)
print("\nSome demand outliers:")
print(
    df[df["demand_outlier"]][
        ["userid", "location", "duration", "demand"]
    ].head()
)


Duration outlier information
Q1: 41.0
Q3: 205.0
IQR: 164.0
Lower limit: -205.0
Upper limit: 451.0
Duration outliers: 4120

Demand outlier information
Q1: 7.69
Q3: 23.2
IQR: 15.509999999999998
Lower limit: -15.574999999999996
Upper limit: 46.464999999999996
Demand outliers: 3528

Some duration outliers:
     userid location  duration  demand
7         0    hotel     494.0   55.17
12        0    hotel     493.0   55.83
49        0    hotel     540.0   61.67
92        0    hotel     540.0   61.64
100       0    hotel     531.0   41.33

Some demand outliers:
     userid location  duration  demand
7         0    hotel     494.0   55.17
12        0    hotel     493.0   55.83
49        0    hotel     540.0   61.67
92        0    hotel     540.0   61.64
102       0    hotel     460.0   51.70


## Step 11 — Preview the current cleaned dataset

In [12]:
df.head()

,userid,chargerid,chargercompany,location,chargertype,startday,starttime,endday,endtime,startdatetime,enddatetime,duration,demand,start_full,end_full,duration_original,duration_calculated,duration_mismatch,duration_outlier,demand_outlier
0,0,1,1,hotel,0,2022-09-15,20:54:02,2022-09-15,23:59:13,2022-09-15 20:54,2022-09-15 23:59,185.0,20.36,2022-09-15 20:54:02,2022-09-15 23:59:13,185,185.0,False,False,False
1,0,1,1,hotel,0,2022-09-14,20:01:05,2022-09-14,21:31:04,2022-09-14 20:01,2022-09-14 21:31,90.0,10.19,2022-09-14 20:01:05,2022-09-14 21:31:04,90,90.0,False,False,False
2,0,1,1,hotel,0,2022-09-14,18:54:30,2022-09-14,19:54:29,2022-09-14 18:54,2022-09-14 19:54,60.0,6.78,2022-09-14 18:54:30,2022-09-14 19:54:29,60,60.0,False,False,False
3,0,1,1,hotel,0,2022-09-29,18:32:51,2022-09-30,0:16:42,2022-09-29 18:32,2022-09-30 0:16,344.0,37.65,2022-09-29 18:32:51,2022-09-30 00:16:42,344,344.0,False,False,False
4,0,1,1,hotel,0,2022-09-25,19:30:15,2022-09-26,0:30:14,2022-09-25 19:30,2022-09-26 0:30,300.0,33.81,2022-09-25 19:30:15,2022-09-26 00:30:14,300,300.0,False,False,False


## Step 12 — Final logical validation of identifiers and chronology

### Checks performed
The notebook checks:
- `end_full < start_full`

In [13]:
end_before_start = df["end_full"] < df["start_full"]
print("end_before_start:",end_before_start.sum() )

end_before_start: 0


## Step 13 — Save the cleaned dataset and run final checks

### What this step does
The cleaned DataFrame is exported as:

`ChargingRecords_cleaned.csv`

The notebook then prints:
- final row count
- missing values after cleaning
- final data types
### Outlier note
The outlier rows remain in the saved dataset because they were only **flagged**, not automatically removed.


In [14]:
# 16. Save the cleaned dataset
# The outlier rows are still included in this file.

df.to_csv("data/ChargingRecords_cleaned.csv", index=False)
print("\nRows after cleaning:", len(df))

# 17. Final checks
print("\nMissing values after cleaning:")
print(df.isnull().sum())
print("\nFinal data types:")
print(df.dtypes)


Rows after cleaning: 72241

Missing values after cleaning:
userid                 0
chargerid              0
chargercompany         0
location               0
chargertype            0
startday               0
starttime              0
endday                 0
endtime                0
startdatetime          0
enddatetime            0
duration               0
demand                 0
start_full             0
end_full               0
duration_original      0
duration_calculated    0
duration_mismatch      0
duration_outlier       0
demand_outlier         0
dtype: int64

Final data types:
userid                          int64
chargerid                       int64
chargercompany                  int64
location               string[python]
chargertype                     int64
startday                       object
starttime                      object
endday                         object
endtime                        object
startdatetime                  object
enddatetime                 